In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

Cargar datos y rehacer los splits EXACTAMENTE igual

In [2]:
df = pd.read_parquet('/home/shared_user/datasets/data_preprocesada.parquet')

df[df.select_dtypes('bool').columns] = df.select_dtypes('bool').astype('int8')
le = LabelEncoder(); df['target'] = le.fit_transform(df['target'])
float_cols = ["year_occ_z","lat_rad","lon_rad","Vict Age","target"]
df[float_cols] = df[float_cols].astype('float32')

X, y = df.drop('target', axis=1), df['target']
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y, test_size=.30, stratify=y, random_state=42)
X_val, X_te, y_val, y_te = train_test_split(
        X_tmp, y_tmp, test_size=.50, stratify=y_tmp, random_state=42)

Cargar ambos modelos entrenados previamente

In [3]:
from joblib import load

lgbm = load("dask_lgbm_full.pkl")   # LightGBM local
logr = load("logreg_best.pkl")      # Logistic Regression

Evaluación en la franja morada (validation)

In [7]:
X_val = X_val.fillna(0)
X_te  = X_te .fillna(0)

In [8]:
from sklearn.metrics import f1_score

f1_lgbm = f1_score(y_val, lgbm.predict(X_val), average='macro')
f1_logr = f1_score(y_val, logr.predict(X_val), average='macro')

print(f"LightGBM  F1_val = {f1_lgbm:.4f}")
print(f"LogReg    F1_val = {f1_logr:.4f}")

champion = lgbm if f1_lgbm >= f1_logr else logr
print("\n  Campeón seleccionado →", type(champion).__name__)

LightGBM  F1_val = 0.4381
LogReg    F1_val = 0.2957

  Campeón seleccionado → LGBMClassifier


Reentrenar el campeón en azul + morado

In [9]:
X_train_plus = pd.concat([X_tr, X_val])
y_train_plus = pd.concat([y_tr, y_val])

champion.fit(X_train_plus, y_train_plus)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.040375 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 704
[LightGBM] [Info] Number of data points in the train set: 854240, number of used features: 50
[LightGBM] [Info] Start training from score -1.853313
[LightGBM] [Info] Start training from score -1.938239
[LightGBM] [Info] Start training from score -4.713678
[LightGBM] [Info] Start training from score -2.099572
[LightGBM] [Info] Start training from score -4.310303
[LightGBM] [Info] Start training from score -2.513703
[LightGBM] [Info] Start training from score -6.436132
[LightGBM] [Info] Start training from score -5.801261
[LightGBM] [Info] Start training from score -2.040331
[LightGBM] [Info] Start training from score -1.685809
[LightGBM] [Info] Start tra

LGBMClassifier(colsample_bytree=0.8, num_leaves=70, num_threads=4,
               random_state=42, reg_lambda=1.0, subsample=0.8, time_out=120,
               tree_learner='data')

Evaluación FINAL en la franja naranja (test) SOLO 1 vez

In [10]:
from sklearn.metrics import classification_report

print("\n=== Test (orange) ===")
print(classification_report(y_te, champion.predict(X_te)))


=== Test (orange) ===
              precision    recall  f1-score   support

         0.0       0.62      0.76      0.69     23625
         1.0       0.44      0.46      0.45     21702
         2.0       0.62      0.66      0.64      1352
         3.0       0.82      0.95      0.88     18468
         4.0       0.38      0.07      0.12      2025
         5.0       0.41      0.59      0.48     12206
         6.0       0.52      0.05      0.08       241
         7.0       0.66      0.22      0.33       456
         8.0       0.49      0.21      0.30     19595
         9.0       0.46      0.52      0.49     27933
        10.0       0.40      0.01      0.02       207
        11.0       0.44      0.27      0.34      9946
        12.0       0.40      0.14      0.21      2187
        13.0       0.58      0.59      0.59     10806

    accuracy                           0.54    150749
   macro avg       0.52      0.39      0.40    150749
weighted avg       0.53      0.54      0.52    150749



Guardar el campeón para producción / inferencia

In [11]:
from joblib import dump
dump(champion, "champion.pkl")
print("Modelo campeón guardado en champion.pkl")

Modelo campeón guardado en champion.pkl
